# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [4]:

# Task 1 — CO2 per capita for all countries, highlighting one country

# Find the per-capita column in the dataset
candidates = [
    'CO2_per_capita',
    'CO2_Per_Capita',
    'CO2_per_capita_tonnes',
    'CO2_per_capita_t'
]

per_capita_col = next(
    (col for col in candidates if col in df.columns),
    None
)

if per_capita_col is None:
    print("Available columns:", df.columns.tolist())
    raise ValueError("Find the correct CO2 per-capita column name above.")

# Choose a country to highlight
highlight = 'United States'

# Grey for all countries except the highlighted country
color_map = {
    country: '#EEEEEE' if country != highlight else '#C0392B'
    for country in df['Country'].unique()
}

fig = px.line(
    df,
    x='Year',
    y=per_capita_col,
    color='Country',
    color_discrete_map=color_map,
    category_orders={
        'Country': [
            c for c in df['Country'].unique() if c != highlight
        ] + [highlight]
    },
    labels={
        'Year': '',
        per_capita_col: 'CO₂ emissions per capita'
    },
    title="The United States Remains a High Per-Capita CO₂ Emitter",
    template='plotly_white',
    height=550
)

# Thin background lines; no legend
fig.update_traces(
    line=dict(width=1),
    showlegend=False
)

# Highlight the selected country
fig.update_traces(
    line=dict(width=3, color='#C0392B'),
    selector=dict(name=highlight)
)

# Label the highlighted line at its final available year
highlight_data = (
    df[df['Country'] == highlight]
    .dropna(subset=[per_capita_col])
    .sort_values('Year')
)

if highlight_data.empty:
    raise ValueError(f"No data found for {highlight}.")

last = highlight_data.iloc[-1]

fig.add_annotation(
    x=last['Year'],
    y=last[per_capita_col],
    text=highlight,
    showarrow=False,
    xanchor='left',
    xshift=8,
    font=dict(color='#C0392B', size=12)
)

fig.update_layout(
    margin=dict(l=60, r=120, t=90, b=50),
    xaxis=dict(showgrid=False, title=''),
    yaxis=dict(gridcolor='#EEEEEE'),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [6]:
# Task — Multi-series line with highlight
# YOUR CODE HERE


# Task 2 — Asian countries, highlighting China

# Select all countries whose region contains "Asia"
asia_df = df[
    df['Region'].str.contains('Asia', case=False, na=False)
].copy()

if asia_df.empty:
    print("Available regions:", df['Region'].unique())
    raise ValueError("Check the region names in the dataset.")

highlight = 'China'

if highlight not in asia_df['Country'].unique():
    raise ValueError("China is not present in the selected Asian countries.")

# Grey context lines and one highlighted country
color_map = {
    country: '#EEEEEE' if country != highlight else '#C0392B'
    for country in asia_df['Country'].unique()
}

fig = px.line(
    asia_df,
    x='Year',
    y='CO2_Mt',
    color='Country',
    color_discrete_map=color_map,
    labels={
        'Year': '',
        'CO2_Mt': 'CO₂ emissions (Mt)'
    },
    title="China's CO₂ Emissions Have Risen Above Other Asian Countries",
    template='plotly_white',
    height=550
)

# Remove the legend and keep background lines thin
fig.update_traces(
    line=dict(width=1),
    showlegend=False
)

# Emphasize China
fig.update_traces(
    line=dict(width=3, color='#C0392B'),
    selector=dict(name=highlight)
)

# Direct label at China's last available data point
china = (
    asia_df[asia_df['Country'] == highlight]
    .dropna(subset=['CO2_Mt'])
    .sort_values('Year')
)

if china.empty:
    raise ValueError("No CO2_Mt data found for China.")

last = china.iloc[-1]

fig.add_annotation(
    x=last['Year'],
    y=last['CO2_Mt'],
    text='China',
    showarrow=False,
    xanchor='left',
    xshift=8,
    font=dict(color='#C0392B', size=12)
)

fig.update_layout(
    margin=dict(l=60, r=100, t=90, b=50),
    xaxis=dict(showgrid=False, title=''),
    yaxis=dict(gridcolor='#EEEEEE'),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()
